In [26]:
#| default_exp models.memtr

In [3]:
#| hide
%load_ext autoreload
%autoreload 2

In [4]:
#| export
import torch, torch.nn as nn, copy, re, torch.nn.functional as F

from typing import Optional, Tuple, List

from transformers import PretrainedConfig, DistilBertConfig, DistilBertPreTrainedModel, DistilBertModel
from transformers.models.distilbert.modeling_distilbert import TransformerBlock

from xcai.losses import *
from xcai.models.PPP0XX import *

from xcai.models.modeling_utils import *

## Load data

In [5]:
from xcai.main import *

In [6]:
data_dir = '/Users/suchith720/Projects/data'
config_file = 'wikiseealsotitles'
config_key = 'data_meta'

mname = 'sentence-transformers/msmarco-distilbert-base-v4'

pkl_dir = '/Users/suchith720/Projects/data/processed/'
pkl_file = f'{pkl_dir}/mogicX/wikiseealsotitles_data-meta_distilbert-base-uncased_sxc.joblib'

In [7]:
block = build_block(pkl_file, config_file, True, config_key, data_dir=data_dir, n_slbl_samples=2, 
                    n_sdata_meta_samples=3, return_scores=True, do_build=True)

/Users/suchith720/Projects/pyxclib/xclib/data/data_utils.py:263: UserWarning: Header mis-match from inferred shape!
  warnings.warn("Header mis-match from inferred shape!")


In [8]:
block.train.dset.meta['neg_meta'] = copy.deepcopy(block.train.dset.meta['cat_meta'])
block.train.dset.meta['neg_meta'].prefix = 'neg'

In [9]:
block.train.dset.data.main_oversample = True

In [10]:
batch = block.train.one_batch(4)

In [11]:
print(list(batch))

['data_idx', 'data_identifier', 'data_input_text', 'data_input_ids', 'data_attention_mask', 'plbl2data_idx', 'plbl2data_data2ptr', 'plbl2data_scores', 'lbl2data_idx', 'lbl2data_scores', 'lbl2data_data2ptr', 'lbl2data_identifier', 'lbl2data_input_text', 'lbl2data_input_ids', 'lbl2data_attention_mask', 'pcat2data_idx', 'pcat2data_data2ptr', 'pcat2data_scores', 'cat2data_idx', 'cat2data_scores', 'cat2data_data2ptr', 'cat2data_identifier', 'cat2data_input_text', 'cat2data_input_ids', 'cat2data_attention_mask', 'pcat2lbl_idx', 'pcat2lbl_lbl2ptr', 'pcat2lbl_scores', 'cat2lbl_idx', 'cat2lbl_scores', 'cat2lbl_lbl2ptr', 'cat2lbl_identifier', 'cat2lbl_input_text', 'cat2lbl_input_ids', 'cat2lbl_attention_mask', 'cat2lbl_data2ptr', 'pcat2lbl_data2ptr', 'pneg2data_idx', 'pneg2data_data2ptr', 'pneg2data_scores', 'neg2data_idx', 'neg2data_scores', 'neg2data_data2ptr', 'neg2data_identifier', 'neg2data_input_text', 'neg2data_input_ids', 'neg2data_attention_mask', 'pneg2lbl_idx', 'pneg2lbl_lbl2ptr', 'pn

In [12]:
batch['lbl2data_data2ptr'], batch['plbl2data_data2ptr']

(tensor([2, 2, 2, 2]), tensor([3, 1, 2, 4]))

In [13]:
batch['data_input_text']

['Anarchism', 'Autism', 'Aristotle', 'Academy Awards']

In [115]:
batch['lbl2data_input_text']

['Libertarian socialism',
 'Libertarian socialism',
 'Autism',
 'Autism',
 'Conimbricenses',
 'Aristotle',
 'List of Academy Award records',
 'Academy Awards']

## `MemConfig`

In [12]:
#| export
class MEMConfig(DBTConfig):

    def __init__(
        self,
        base_model_dim:Optional[int]=None,
        num_train_data:Optional[int]=None,
        num_test_data:Optional[int]=None,
        num_labels:Optional[int]=None,
        num_metadata:Optional[int]=None,
        data_aug_meta_prefix:Optional[str]=None,
        combiner_heads:Optional[int]=16,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.base_model_dim, self.num_train_data, self.num_test_data = base_model_dim, num_train_data, num_test_data
        self.num_labels, self.num_metadata, self.data_aug_meta_prefix = num_labels, num_metadata, data_aug_meta_prefix
        self.combiner_heads = combiner_heads
        

## Combiner block

In [13]:
#| export
class CrossCombinerBlock(TransformerBlock):

    def __init__(self, config:PretrainedConfig):
        super().__init__(config)

    def post_init(self):
        for module in self.modules(): self._init_weights(module)

    def _initialize_weights(self, module: nn.Module):
        for m in module.modules(): self._init_weights(m)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            torch.nn.init.eye_(module.weight)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def forward(
        self,
        x: torch.Tensor,
        m: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        head_mask: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
    ) -> Tuple[torch.Tensor, ...]:
        
        # Cross-Attention
        ca_output = self.attention(
            query=x,
            key=m,
            value=m,
            mask=attn_mask,
            head_mask=head_mask,
            output_attentions=output_attentions,
        )
        if output_attentions:
            ca_output, ca_weights = ca_output  # (bs, seq_length, dim), (bs, n_heads, seq_length, seq_length)
        else:  # To handle these `output_attentions` or `output_hidden_states` cases returning tuples
            if type(ca_output) is not tuple:
                raise TypeError(f"ca_output must be a tuple but it is {type(ca_output)} type")
            ca_output = ca_output[0]
            
        ca_output = self.sa_layer_norm(ca_output + x)  # (bs, seq_length, dim)

        # Feed Forward Network
        ffn_output = self.ffn(ca_output)  # (bs, seq_length, dim)
        ffn_output: torch.Tensor = self.output_layer_norm(ffn_output + ca_output)  # (bs, seq_length, dim)

        output = (ffn_output,)
        if output_attentions:
            output = (ca_weights,) + output
        return output
        

## Encoder

In [27]:
#| export
class Encoder(DistilBertPreTrainedModel):
    
    def __init__(
        self, 
        config
    ):
        super().__init__(config)
        self.config = config
        self.distilbert = DistilBertModel(config)
        
        self.projector = nn.Linear(config.dim, config.base_model_dim)
        
        config.dim, config.n_heads = config.base_model_dim, config.combiner_heads
        self.combiner = CrossCombinerBlock(config)
        
    @torch.no_grad()
    def init_dr_head(self):
        torch.nn.init.eye_(self.projector.weight)

    def _init_weights(self, module: nn.Module):
        super()._init_weights(module)
        if isinstance(module, Encoder):
            module.init_dr_head()

    def fuse_metadata(self, data_rep:torch.Tensor, meta_rep:torch.Tensor, data2ptr:torch.Tensor):
        num_metadata = data2ptr.max()
        assert (
            (data2ptr == num_metadata).all()
        ), "All the queries should have the same number of metadata."
        
        data_rep, meta_rep = data_rep.unsqueeze(1), meta_rep.view(data_rep.shape[0], num_metadata, -1)
        fused_data_rep = self.combiner(data_rep, meta_rep)[0]
        fused_data_rep = fused_data_rep.squeeze(1)
        return F.normalize(fused_data_rep, dim=1)

    def get_metadata_representation(
        self,
        data_input_ids:Optional[torch.Tensor]=None,
        data_attention_mask:Optional[torch.Tensor]=None,
        **kwargs,
    ):
        o = self.distilbert(
            input_ids=data_input_ids,
            attention_mask=data_attention_mask,
            **kwargs
        )
        meta_rep = Pooling.mean_pooling(o[0], data_attention_mask)
        return F.normalize(self.projector(meta_rep), dim=1)
            
    def forward(
        self,
        data_rep:Optional[torch.Tensor] = None,
        data_aug_meta_prefix: Optional[str]=None,
        **kwargs
    ):
        data_rep = F.normalize(data_rep, dim=1)

        fused_data_rep = meta_rep = None
        if data_aug_meta_prefix is not None:
            meta_kwargs = Parameters.from_meta_aug_prefix(data_aug_meta_prefix, **kwargs)            
            meta_rep = self.get_metadata_representation(meta_kwargs[data_aug_meta_prefix]["input_ids"],
                                                        meta_kwargs[data_aug_meta_prefix]["attention_mask"])
            fused_data_rep = self.fuse_metadata(data_rep, meta_rep, meta_kwargs[data_aug_meta_prefix]["data2ptr"])
        
        return data_rep, fused_data_rep, meta_rep
        

## `MEM001`

In [33]:
#| export
class MEM001(DistilBertPreTrainedModel):
    use_generation,use_representation = False,True
    _tied_weights_keys = ["encoder.distilbert"]
    
    def __init__(
        self,
        config,
    ):
        super().__init__(config)
        self.config = config
        self.encoder = Encoder(config)

        self.trn_embeds_shards = []
        self.tst_embeds_shards = []
        self.lbl_embeds_shards = []
        
        loss_kwargs = {
            'margin': config.margin, 'n_negatives': config.num_negatives, 'tau': config.tau, 
            'apply_softmax': config.apply_softmax, 'reduce': config.reduction,
        }
        self.loss_fn = get_loss_function(config.loss_function)(**loss_kwargs)
        self.post_init()

    def _shard_tensor(self, x: torch.Tensor) -> List[torch.Tensor]:
        if not torch.cuda.is_available():
            return [x.to("cpu")]
        
        num_gpus = torch.cuda.device_count()
        if num_gpus == 0:
            return [x.to("cpu")]
            
        chunk_size = (x.shape[0] + num_gpus - 1) // num_gpus
        shards = []
        for i in range(num_gpus):
            start = i * chunk_size
            end = min((i + 1) * chunk_size, x.shape[0])
            if start < x.shape[0]:
                shard = x[start:end].to(f"cuda:{i}")
                shards.append(shard)
        return shards

    def _fetch_from_shards(self, shards: List[torch.Tensor], idx: torch.Tensor) -> torch.Tensor:
        if len(shards) == 0:
            raise ValueError("Shards list is empty")
        
        if len(shards) == 1:
            return shards[0][idx].to(idx.device)
            
        chunk_size = shards[0].shape[0]
        shard_indices = idx // chunk_size
        local_indices = idx % chunk_size
        
        out = torch.empty((idx.shape[0], shards[0].shape[1]), dtype=shards[0].dtype, device=idx.device)
        
        unique_shards = torch.unique(shard_indices)
        for s in unique_shards:
            s_item = s.item()
            shard_device = shards[s_item].device
            mask = (shard_indices == s)
            if mask.any():
                local_idx_s = local_indices[mask].to(shard_device)
                out[mask] = shards[s_item][local_idx_s].to(idx.device)
        return out

    def _init_weights(self, module: nn.Module):
        super()._init_weights(module)
        if isinstance(module, BaseLoss):
            module.init_weights()
            
    def _tie_weights(self):
        self.distilbert = self.encoder.distilbert

    @torch.no_grad()
    def init_encoder_embeddings(self, trn_embeds:torch.Tensor, tst_embeds:torch.Tensor, lbl_embeds:torch.Tensor):
        self.trn_embeds_shards = self._shard_tensor(trn_embeds)
        self.tst_embeds_shards = self._shard_tensor(tst_embeds)
        self.lbl_embeds_shards = self._shard_tensor(lbl_embeds)

    def get_label_representation(self, data_idx:Optional[torch.Tensor]=None):
        encoder = nn.DataParallel(module=self.encoder) if self.config.use_encoder_parallel else self.encoder
        lbl_raw = self._fetch_from_shards(self.lbl_embeds_shards, data_idx)
        data_repr, fused_data_repr, _  = encoder(data_rep=lbl_raw)
        return XCModelOutput(
            data_fused_repr=fused_data_repr,
            data_repr=data_repr,
        )

    def forward(
        self,
        data_idx:Optional[torch.Tensor]=None,
        
        lbl2data_data2ptr:Optional[torch.Tensor]=None,
        lbl2data_idx:Optional[torch.Tensor]=None,
        
        neg2data_data2ptr:Optional[torch.Tensor]=None,
        neg2data_idx:Optional[torch.Tensor]=None,

        plbl2data_data2ptr:Optional[torch.Tensor]=None,
        plbl2data_idx:Optional[torch.Tensor]=None,
        
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        **kwargs
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        encoder = nn.DataParallel(module=self.encoder) if self.config.use_encoder_parallel else self.encoder

        data_meta_kwargs = Parameters.from_feat_meta_aug_prefix('data', self.config.data_aug_meta_prefix, **kwargs)

        if getattr(self, "training", False) or getattr(self, "train", False) is True: 
            data_raw = self._fetch_from_shards(self.trn_embeds_shards, data_idx)
        else:
            data_raw = self._fetch_from_shards(self.tst_embeds_shards, data_idx)
            
        data_repr, fused_data_repr, _ = encoder(data_rep=data_raw, data_aug_meta_prefix=self.config.data_aug_meta_prefix, 
                                                **data_meta_kwargs)
        
        loss = lbl2data_repr = neg2data_repr = None
        if lbl2data_idx is not None and neg2data_idx is not None:
            lbl2data_raw = self._fetch_from_shards(self.lbl_embeds_shards, lbl2data_idx)
            neg2data_raw = self._fetch_from_shards(self.lbl_embeds_shards, neg2data_idx)

            lbl2data_repr, _, _ = encoder(data_rep=lbl2data_raw)
            neg2data_repr, _, _ = encoder(data_rep=neg2data_raw)
            
            loss = self.loss_fn(fused_data_repr, pos_targ=lbl2data_repr, n_pos=lbl2data_data2ptr, pos_idx=lbl2data_idx, 
                                neg_targ=neg2data_repr, n_neg=neg2data_data2ptr, neg_idx=neg2data_idx, 
                                n_ppos=plbl2data_data2ptr, ppos_idx=plbl2data_idx, **kwargs)
            
        if not return_dict:
            o = (data_repr, lbl2data_repr)
            return ((loss,) + o) if loss is not None else o

        return XCModelOutput(
            loss=loss,
            data_fused_repr=fused_data_repr,
            data_repr=data_repr,
            lbl2data_repr=lbl2data_repr,
            neg2data_repr=neg2data_repr,
        )
        

### Example

In [32]:
config = MEMConfig(
    num_train_data=block.train.dset.n_data,
    num_test_data=block.test.dset.n_data,
    num_labels=block.train.dset.n_lbl,
    num_metadata=block.train.dset.meta["cat_meta"].n_meta,
    data_aug_meta_prefix="cat2data",
    base_model_dim=12,
    
    margin = 0.3,
    num_negatives = 10,
    tau = 0.1,
    apply_softmax = True,
    reduction = "mean",

    normalize = True,
    use_layer_norm = True,
    
    use_encoder_parallel = False,
    loss_function = "ranking",
)

In [158]:
m = MEM001.from_pretrained('distilbert-base-uncased', config=config)

Some weights of MEM001 were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['encoder.combiner.attention.k_lin.bias', 'encoder.combiner.attention.k_lin.weight', 'encoder.combiner.attention.out_lin.bias', 'encoder.combiner.attention.out_lin.weight', 'encoder.combiner.attention.q_lin.bias', 'encoder.combiner.attention.q_lin.weight', 'encoder.combiner.attention.v_lin.bias', 'encoder.combiner.attention.v_lin.weight', 'encoder.combiner.ffn.lin1.bias', 'encoder.combiner.ffn.lin1.weight', 'encoder.combiner.ffn.lin2.bias', 'encoder.combiner.ffn.lin2.weight', 'encoder.combiner.output_layer_norm.bias', 'encoder.combiner.output_layer_norm.weight', 'encoder.combiner.sa_layer_norm.bias', 'encoder.combiner.sa_layer_norm.weight', 'encoder.lbl_embeds', 'encoder.projector.bias', 'encoder.projector.weight', 'encoder.trn_embeds', 'encoder.tst_embeds', 'loss_fn.ptau']
You should probably TRAIN this model on a down-stream task to be able to use it for predict

In [159]:
o = m(**batch)

In [161]:
o.loss

tensor(4.0700, grad_fn=<DivBackward0>)

In [60]:
def func():
    o = m(**batch)
    